# Native Keras evaluation
Image-level text labels only. Dummy results do not establish OCR accuracy or deployment performance.

Report literal complete-string exact only: case, spaces, symbols and order matter. Historical CCT plate-engine replay verifies2,315/2,693 and 644/984, but these are different weights/cohorts from dummy evaluation. The public 37-class evaluator uses linear resizing and first-PAD termination; current 41-class price research uses BICUBIC and removes every PAD, so its 66/144 result is not inherited.

Set `USE_ENERGY_METER = True` if you trained the meter example.

In [ ]:
from pathlib import Path
import os, subprocess, sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
os.chdir(ROOT)
assert (ROOT / 'config/cct_s_v2_global_plate_config.yaml').exists()
def run(*args): subprocess.run(list(args), check=True)

USE_ENERGY_METER = os.environ.get("CCT_ENERGY_METER", "0") == "1"
WORK = Path(os.environ.get('CCT_WORK', 'artifacts/energy_meter' if USE_ENERGY_METER else 'artifacts'))
TRAINING = Path(os.environ.get("CCT_OUTPUT", WORK if USE_ENERGY_METER else WORK / "training"))
PLATE_CONFIG = ('config/cct_s_energy_meter_config.yaml' if USE_ENERGY_METER
                else 'config/cct_s_v2_global_plate_config.yaml')
ANNOTATIONS = ('dataset/energy-meter/val_annotations.csv' if USE_ENERGY_METER
               else 'dummy_data/val_annotations.csv')


In [ ]:
checkpoints = list(TRAINING.rglob('best.keras'))
if not checkpoints:
    raise FileNotFoundError('Run 02_train.ipynb first.')
MODEL = str(max(checkpoints, key=lambda p: p.stat().st_mtime))
run(sys.executable,'scripts/evaluate.py','--annotations',ANNOTATIONS,'--plate-config',PLATE_CONFIG,'--backend','native','--model',MODEL,'--output',str(WORK / 'native.json'))